In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

df_panel = pd.read_csv("C:\\Users\\rohin\\Sucide Project\\Panel Data Construction\\panel_dataset_ready.csv")
df_sa = pd.read_csv("C:\\Users\\rohin\\Sucide Project\\Panel Data Construction\\df_sa_panel_dataset_ready.csv")


In [17]:
features = [
    'AlcoholRate_roll3',
    'HomicideRate_roll3',
    'GDP_per_capita_roll3'
]

In [18]:
print(df_panel[features].head())
print(df_sa[features].head())


   AlcoholRate_roll3  HomicideRate_roll3  GDP_per_capita_roll3
0           0.004656           10.445060            162.856268
1           0.004656           10.536500            150.781545
2           0.004656           10.597834            138.706822
3           0.004832           10.618478            152.122577
4           0.005008           10.816649            165.538333
   AlcoholRate_roll3  HomicideRate_roll3  GDP_per_capita_roll3
0           8.742056           44.599059           3094.281929
1           8.742056           44.648861           2970.716924
2           8.742056           44.673990           2847.151919
3           8.655983           44.664608           2794.180200
4           8.569911           46.262092           2741.208480


In [19]:
split_idx = int(len(df_sa) * 0.8)

X_train_sa = df_sa.iloc[:split_idx][features]
y_train_sa = df_sa.iloc[:split_idx]['SuicideRate_diff']

X_test_sa = df_sa.iloc[split_idx:][features]
y_test_sa = df_sa.iloc[split_idx:]['SuicideRate_diff']


In [20]:
sa_only_model = Ridge(alpha=100)
sa_only_model.fit(X_train_sa, y_train_sa)

sa_only_preds = sa_only_model.predict(X_test_sa)
sa_only_mae = mean_absolute_error(y_test_sa, sa_only_preds)

sa_only_mae


0.3689999181564017

In [22]:
df_train_panel = df_panel[df_panel['Country'] != 'South Africa']

X_panel = df_train_panel[features]
y_panel = df_train_panel['SuicideRate_diff']


In [23]:
panel_model = Ridge(alpha=100)
panel_model.fit(X_panel, y_panel)


Ridge(alpha=100)

In [24]:
panel_preds = panel_model.predict(X_test_sa)
panel_mae = mean_absolute_error(y_test_sa, panel_preds)

panel_mae


0.34570048873857623

In [25]:
print(f"SA-only MAE:   {sa_only_mae:.4f}")
print(f"Panel MAE:     {panel_mae:.4f}")

SA-only MAE:   0.3690
Panel MAE:     0.3457


In [26]:
panel_residuals = y_panel - panel_model.predict(X_panel)
residual_std = panel_residuals.std()

residual_std


0.527141491171442

In [27]:
df_sa_test = df_sa.iloc[split_idx:].copy()

df_sa_test['Predicted_diff'] = panel_preds
df_sa_test['Lower_95'] = df_sa_test['Predicted_diff'] - 1.96 * residual_std
df_sa_test['Upper_95'] = df_sa_test['Predicted_diff'] + 1.96 * residual_std

df_sa_test[['Year', 'Predicted_diff', 'Lower_95', 'Upper_95']]


,Year,Predicted_diff,Lower_95,Upper_95
50,2017,-0.029329,-1.062526,1.003868
51,2018,-0.029705,-1.062903,1.003492
52,2018,-0.031203,-1.064400,1.001994
53,2018,-0.032117,-1.065314,1.001081
54,2019,-0.030834,-1.064031,1.002363
55,2019,-0.029670,-1.062867,1.003527
56,2019,-0.028440,-1.061638,1.004757
57,2020,-0.028226,-1.061424,1.004971
58,2020,-0.028622,-1.061820,1.004575
59,2020,-0.028700,-1.061897,1.004497


In [28]:
final_model = Ridge(alpha=100)
final_model.fit(X_panel, y_panel)

Ridge(alpha=100)

In [31]:
def predict_next_suicide_rate(
    last_rate,
    alcohol_roll3,
    homicide_roll3,
    gdp_roll3
):
    X = pd.DataFrame(
        [[alcohol_roll3, homicide_roll3, gdp_roll3]],
        columns=[
            'AlcoholRate_roll3',
            'HomicideRate_roll3',
            'GDP_per_capita_roll3'
        ]
    )
    
    delta = final_model.predict(X)[0]
    return last_rate + delta

In [33]:
example_prediction = predict_next_suicide_rate(
    last_rate=22.3,
    alcohol_roll3=7.8,
    homicide_roll3=12.53,
    gdp_roll3=65234
)

example_prediction

22.266074715751703

In [34]:
df_panel['GDP_per_capita_roll3'].describe()


count      9862.000000
mean      11905.953251
std       18208.682221
min         109.593814
25%        1315.880956
50%        4238.721628
75%       13213.244968
max      134965.815442
Name: GDP_per_capita_roll3, dtype: float64